In [2]:
import pandas as pd

path = "s3://vivek-mlproject1-capstone-1nov2026/raw/bank-additional-full.csv"
df = pd.read_csv(path, sep=";")
df.shape

(41188, 21)

In [3]:
df.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,duration,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,261,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,149,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,226,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,151,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,307,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


In [4]:
df.dtypes

age                 int64
job                object
marital            object
education          object
default            object
housing            object
loan               object
contact            object
month              object
day_of_week        object
duration            int64
campaign            int64
pdays               int64
previous            int64
poutcome           object
emp.var.rate      float64
cons.price.idx    float64
cons.conf.idx     float64
euribor3m         float64
nr.employed       float64
y                  object
dtype: object

In [5]:
df["y"].value_counts(normalize=True)
df["default"].value_counts()
df.groupby("job")["y"].apply(lambda s: (s == "yes").mean()).sort_values(ascending=False)

job
student          0.314286
retired          0.252326
unemployed       0.142012
admin.           0.129726
management       0.112175
unknown          0.112121
technician       0.108260
self-employed    0.104856
housemaid        0.100000
entrepreneur     0.085165
services         0.081381
blue-collar      0.068943
Name: y, dtype: float64

In [6]:
df["y"].value_counts(normalize=True)

y
no     0.887346
yes    0.112654
Name: proportion, dtype: float64

In [7]:
df["default"].value_counts()

default
no         32588
unknown     8597
yes            3
Name: count, dtype: int64

In [8]:
df.groupby("job")["y"].agg(rate=lambda s: (s == "yes").mean(), n="count").sort_values("rate", ascending=False)

,rate,n
job,,
student,0.314286,875
retired,0.252326,1720
unemployed,0.142012,1014
admin.,0.129726,10422
management,0.112175,2924
unknown,0.112121,330
technician,0.108260,6743
self-employed,0.104856,1421
housemaid,0.100000,1060


In [9]:
df2 = df.copy()

# 1. target to 1/0
df2["y"] = (df2["y"] == "yes").astype(int)

# 2. drop the leaky column
df2 = df2.drop(columns=["duration"])

# 3. pdays: 999 means "never contacted" -> flag + neutral value
df2["pdays_contacted"] = (df2["pdays"] != 999).astype(int)
df2["pdays"] = df2["pdays"].where(df2["pdays"] != 999, 0)

# 4. one-hot encode the text columns
cat_cols = df2.select_dtypes("object").columns
df2 = pd.get_dummies(df2, columns=cat_cols, dtype=int)

print(df2.shape)
print(df2["y"].mean())

(41188, 64)
0.11265417111780131


In [10]:
from sklearn.model_selection import train_test_split

bucket = "vivek-mlproject1-capstone-1nov2026"

# target first, then features (the layout SageMaker's XGBoost expects)
cols = ["y"] + [c for c in df2.columns if c != "y"]
df2 = df2[cols]

# 70% train, then split the remaining 30% evenly into validation and test
train, temp = train_test_split(df2, test_size=0.30, stratify=df2["y"], random_state=42)
val, test = train_test_split(temp, test_size=0.50, stratify=temp["y"], random_state=42)

for name, part in [("train", train), ("validation", val), ("test", test)]:
    print(name, part.shape, round(part["y"].mean(), 4))
    part.to_csv(f"s3://{bucket}/processed/{name}.csv", index=False, header=False)

# keep the column names, since the CSVs have no header
pd.Series(cols).to_csv(f"s3://{bucket}/processed/columns.csv", index=False, header=False)

train (28831, 64) 0.1127
validation (6178, 64) 0.1127
test (6179, 64) 0.1126
